# P7 RAG + QLoRA on SQuAD 2.0: Kaggle runner

Thin wrapper: all logic lives in `src/ragqa`.

Settings: **GPU T4, Internet ON**, then *Save Version -> Save & Run All*. Every stage is resumable: rerunning skips
retrieval rows, prediction files and adapters that already exist. Each config runs with `check=True`, so a failing
smoke run stops the notebook before the long run.

Download `p7_results.zip` from the Output tab: retrieval ablation, predictions, results, bootstrap, error analysis,
fine-tuning logs and the environment. LoRA adapters (~70 MB each) are left out. SQuAD 2.0 is CC BY-SA 4.0: files that
quote passages or questions are share-alike.

In [ ]:
!git clone --depth 1 https://github.com/JDevender007/Artificial_Intelligence.git
%cd Artificial_Intelligence/07-rag-qlora
!pip install -q -r requirements-gpu.txt && pip install -q -e . --no-deps
# Kaggle ships torchao 0.10; peft >= 0.18 refuses to load adapters next to torchao < 0.16. Not used here.
!pip uninstall -q -y torchao

In [ ]:
!mkdir -p /kaggle/working/outputs
!python -c "import torch, transformers, peft, bitsandbytes, accelerate, numpy, pandas, scipy, yaml; [print(m.__name__, m.__version__) for m in (torch, transformers, peft, bitsandbytes, accelerate, numpy, pandas, scipy, yaml)]; print('cuda', torch.cuda.is_available(), torch.version.cuda)" | tee /kaggle/working/outputs/env.txt
!(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo no-gpu) | tee -a /kaggle/working/outputs/env.txt
!pip freeze | grep -iE '^(torch|transformers|peft|bitsandbytes|accelerate|tokenizers|safetensors|huggingface.hub|numpy|pandas|scipy|pyyaml|tabulate)==' | tee /kaggle/working/outputs/versions.txt

## Run `configs/smoke.yaml`

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/run_experiment.py', '--config', 'configs/smoke.yaml', '--set', 'output_dir=/kaggle/working/outputs/smoke', 'data.raw_dir=/kaggle/working/data'], check=True)

In [ ]:
from pathlib import Path

for f in ['data_summary.json', 'retrieval.md', 'retrieval_choice.json', 'results.md', 'bootstrap.md', 'error_analysis.md']:
    p = Path('/kaggle/working/outputs/smoke') / f
    print(f'===== {f}')
    print(p.read_text() if p.exists() else '(missing)')

## Run `configs/squad.yaml`

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'scripts/run_experiment.py', '--config', 'configs/squad.yaml', '--set', 'output_dir=/kaggle/working/outputs/squad', 'data.raw_dir=/kaggle/working/data'], check=True)

In [ ]:
from pathlib import Path

for f in ['data_summary.json', 'retrieval.md', 'retrieval_choice.json', 'results.md', 'bootstrap.md', 'error_analysis.md']:
    p = Path('/kaggle/working/outputs/squad') / f
    print(f'===== {f}')
    print(p.read_text() if p.exists() else '(missing)')

## Package results (adapters and raw data left out)

In [ ]:
%cd /kaggle/working
!zip -qr p7_results.zip outputs -x '*/adapters/*'
!rm -rf outputs data Artificial_Intelligence && ls -la /kaggle/working